# Union-find over the identified graph

Times a full connected-components pass over the local `works_identified` table, the way a batch matcher would run it every 15 minutes.

In [1]:
import time
import numpy as np
import polars as pl

from adapters.utils.iceberg import LocalIcebergTableConfig, get_local_table

table = get_local_table(
    LocalIcebergTableConfig(table_name="works_identified", namespace="matcher", db_name="matcher_catalog"),
    create_if_not_exists=False,
)

started = time.time()
works = pl.from_arrow(table.scan(selected_fields=("id", "version", "type", "merge_candidate_ids")).to_arrow())
print(f"read {works.height:,} rows in {time.time() - started:.1f}s")

read 3,209,373 rows in 0.2s


In [2]:
started = time.time()
edges = (
    works.select("id", "merge_candidate_ids")
    .explode("merge_candidate_ids")
    .drop_nulls()
    .rename({"id": "src", "merge_candidate_ids": "dst"})
    .filter(pl.col("src") != pl.col("dst"))
)
suppressed = works.filter(pl.col("type") == "Deleted").select("id")
edges = edges.join(suppressed, left_on="src", right_on="id", how="anti").join(suppressed, left_on="dst", right_on="id", how="anti")

node_ids = pl.concat([works.select("id"), edges.select(pl.col("dst").alias("id"))]).unique().sort("id")
placeholders = node_ids.height - works.height
index = {w: i for i, w in enumerate(node_ids["id"].to_list())}
src = np.fromiter((index[s] for s in edges["src"]), dtype=np.int64, count=edges.height)
dst = np.fromiter((index[d] for d in edges["dst"]), dtype=np.int64, count=edges.height)
print(f"{node_ids.height:,} nodes ({placeholders:,} placeholders), {edges.height:,} edges, built in {time.time() - started:.1f}s")

3,463,949 nodes (254,576 placeholders), 1,190,885 edges, built in 3.1s


In [3]:
def union_find_python(n: int, src: np.ndarray, dst: np.ndarray) -> np.ndarray:
    parent = list(range(n))

    def find(x: int) -> int:
        root = x
        while parent[root] != root:
            root = parent[root]
        while parent[x] != root:
            parent[x], x = root, parent[x]
        return root

    for a, b in zip(src.tolist(), dst.tolist()):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)
    return np.fromiter((find(i) for i in range(n)), dtype=np.int64, count=n)


def union_find_numpy(n: int, src: np.ndarray, dst: np.ndarray) -> np.ndarray:
    label = np.arange(n)
    while True:
        lo = np.minimum(label[src], label[dst])
        new = label.copy()
        np.minimum.at(new, src, lo)
        np.minimum.at(new, dst, lo)
        new = new[new]
        if np.array_equal(new, label):
            return label
        label = new


for name, fn in [("python", union_find_python), ("numpy", union_find_numpy)]:
    started = time.time()
    labels = fn(node_ids.height, src, dst)
    print(f"{name:>7}: {time.time() - started:.1f}s, {len(np.unique(labels)):,} components")

 python: 1.8s, 2,371,184 components


  numpy: 0.3s, 2,371,184 components


In [4]:
components = pl.DataFrame({"id": node_ids["id"], "component": labels}).join(works.select("id", "type"), on="id", how="left")
sizes = components.group_by("component").len().sort("len", descending=True)
print("components with more than one node:", sizes.filter(pl.col("len") > 1).height)
print("size distribution:", sizes.group_by("len").agg(pl.len().alias("components")).sort("len").to_dict(as_series=False))
for row in sizes.head(5).iter_rows(named=True):
    members = components.filter(pl.col("component") == row["component"]).sort("id")
    print(row["len"], members["id"].to_list()[:12], members["type"].value_counts().to_dict(as_series=False))

components with more than one node: 621751
size distribution: {'len': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 42, 43, 44, 45, 46, 47, 49, 53, 55, 56, 59, 62, 63, 69, 72, 82, 86, 88, 92, 101, 104, 137, 146, 160, 203], 'components': [1749433, 267213, 257298, 85962, 9990, 440, 226, 138, 85, 65, 57, 40, 33, 21, 15, 12, 10, 15, 8, 7, 10, 12, 5, 4, 12, 7, 4, 4, 1, 2, 3, 2, 3, 1, 2, 1, 3, 3, 2, 2, 2, 1, 3, 2, 3, 1, 2, 1, 1, 1, 2, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
203 ['a7sz479b', 'a93jgfeu', 'ac3rvqm6', 'afrche5t', 'aq9wtcwb', 'aqj474dz', 'aqwakead', 'atapuygr', 'atyhqkcx', 'atzk4hkv', 'av73mhsm', 'av7pzvme'] {'type': ['Visible'], 'count': [203]}
160 ['aey5kvce', 'aq9pj88k', 'az22k34y', 'b9t3s8eb', 'bbgqhxku', 'bjfqppt2', 'bv26q7vu', 'bz44wu59', 'c3hgrg7n', 'c5qfq42g', 'ce9wfved', 'cg99c6ws'] {'type': ['Invisible', 'Visible'], 'count': [1, 159]}
146 ['a4yn5spe', 'a8dza7qt', 'a8hp6